# C2 - StudentLife MODEL TABLE + TARGET + SPLIT (step 3)

**Purpose:** build the table a model can learn from, **without data leakage**, and decide who is in the train and test groups. **No model is trained here** (that is step 4, after we check this).

## Decisions made for this PRELIMINARY experiment (please keep them in the report)
1. **Target (reference label from real EMA answers, not an invented C2 label):**
   - `y_stressed = 1` when the person answered stress level **2** ("Definitely stressed") or **3** ("Stressed out").
   - `y_stressed = 0` when they answered **4** ("Feeling good") or **5** ("Feeling great").
   - Level **1** ("A little stressed") is **excluded** because it is ambiguous (not clearly stressed, not clearly good).
   - This is a **stress** label, NOT a C2 capacity label. It is only a way to practise the pipeline. Final C2 labels must come from our own working-student data.
2. **Inputs are phone-sensing only** (your C2 design: no self-report as input). Self-report is used only as the target.
3. **No leakage:** for each stress answer, features use ONLY the **24 hours BEFORE** the answer time. Nothing after it.
4. **Sleep proxy is NOT used** (its correlation with reported sleep was only 0.098 in step 2).
5. **Personal baseline (your key idea):** for some features we also add `dev_<name>` = today's value minus the average of that same person's EARLIER windows (needs at least 3 earlier answers). It only looks backwards in time, so it does not leak.
6. **Participant ID is only used to keep one person's rows together in train OR test. It is never a feature.**

StudentLife only. Not mixed with SWELL-KW.

## Cell 1 - Connect Drive, find data, load the saved stress answers
Needs the files saved by notebook 2 (`studentlife_ema_stress_long.csv`). Click **Connect to Google Drive** if the box appears.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json
import numpy as np
import pandas as pd

def find_data_root(start='/content/drive/MyDrive', max_depth=10):
    start_depth = start.rstrip('/').count('/')
    for dirpath, dirnames, _ in os.walk(start):
        if dirpath.count('/') - start_depth >= max_depth:
            dirnames[:] = []
            continue
        if os.path.basename(dirpath) == 'sensing' and 'phonelock' in dirnames:
            return Path(dirpath).parent
    return None

DATA_ROOT = find_data_root()
assert DATA_ROOT is not None, 'StudentLife folder not found.'
sensing_dir = DATA_ROOT / 'sensing'
OUT_DIR = Path('/content/drive/MyDrive/AnchorAI_c2_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

stress = pd.read_csv(OUT_DIR / 'studentlife_ema_stress_long.csv')
print('stress answers loaded:', len(stress), '| columns:', list(stress.columns))

## Cell 2 - Build the target from real answers
Maps the stress level code to `y_stressed` using the decision above. Level 1 and missing answers are dropped and **counted**, never hidden.

In [ ]:
stress = stress.dropna(subset=['stress_level_code']).copy()
stress['stress_level_code'] = stress['stress_level_code'].astype(int)
print('answers per level:\n', stress['stress_level_code'].value_counts().sort_index())

target_map = {2: 1, 3: 1, 4: 0, 5: 0}          # level 1 deliberately not in the map
answers = stress[stress['stress_level_code'].isin(target_map)].copy()
answers['y_stressed'] = answers['stress_level_code'].map(target_map)
print('\nkept:', len(answers), '| excluded (level 1):', int((stress['stress_level_code'] == 1).sum()))
print('class balance:\n', answers['y_stressed'].value_counts())
print('participants with kept answers:', answers['uid'].nunique())
answers = answers[['uid', 'resp_time', 'stress_level_code', 'y_stressed']].sort_values(['uid', 'resp_time']).reset_index(drop=True)

## Cell 3 - Helper functions (same ideas as notebook 2, now in "minutes since start" form)
For each participant we make a **minute-by-minute grid** from their first to last timestamp. For every sensing source we mark which minutes were active.
Then "what happened in the 24 hours before an answer" is just a slice of that grid, which is fast and cannot look into the future.

In [ ]:
TZ = 'America/New_York'
WINDOW_MIN = 24 * 60            # look back 24 hours (decision 3)
MAX_INTERVAL_DAYS = 7
ACT_CODES = [0, 1, 2, 3]        # activity codes seen in notebook 2

def uid_from_name(name):
    m = re.search(r'(u\d{2})', name)
    return m.group(1) if m else None

def files_by_uid(folder, ext='.csv'):
    out = {}
    for f in sorted(Path(folder).glob('*' + ext)):
        uid = uid_from_name(f.name)
        if uid:
            out.setdefault(uid, []).append(f)
    return out

def read_intervals(paths):
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    df.columns = df.columns.str.strip()
    df = df[['start', 'end']].dropna().drop_duplicates()      # u20 has a duplicate charge file
    good = (df['end'] > df['start']) & ((df['end'] - df['start']) <= MAX_INTERVAL_DAYS * 86400)
    return df[good].sort_values('start').reset_index(drop=True)

def minute_mask(intervals, t0, n):
    """Boolean array: True for every minute (counted from t0) that falls inside an interval."""
    m = np.zeros(n, dtype=bool)
    for s, e in intervals[['start', 'end']].values:
        a = max(int((s - t0) // 60), 0)
        b = min(int((e - t0) // 60) + 1, n)
        if b > a:
            m[a:b] = True
    return m

def read_activity(paths):
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    df.columns = df.columns.str.strip()
    df = df[['timestamp', 'activity inference']].dropna().drop_duplicates().sort_values('timestamp')
    return df['timestamp'].values, df['activity inference'].astype(int).values

APP_PKG_COL = 'RUNNING_TASKS_topActivity_mPackage'
def read_apps(paths):
    df = pd.concat([pd.read_csv(p, usecols=['timestamp', APP_PKG_COL]) for p in paths], ignore_index=True)
    df = df.dropna().sort_values('timestamp')
    codes, _ = pd.factorize(df[APP_PKG_COL])     # app names are turned into anonymous numbers right away
    return df['timestamp'].values, codes
print('helpers ready')

## Cell 4 - Build one row per stress answer (the slow cell)
For each participant: load their sensing files once, build the minute grid, then for each of their answers compute features from the 24 hours **before** the answer.

Features per answer (all from the window BEFORE the answer):
- `lock_min`, `dark_min`, `charge_min` - active minutes in the window
- `night_lock_min`, `night_dark_min`, `night_charge_min` - same but only local 22:00-06:00
- `act_frac_0..3`, `act_n` - activity code shares and how many samples (also a sensing-coverage hint)
- `app_n_snapshots`, `app_n_distinct` - app snapshots and distinct apps

In [ ]:
lock_f   = files_by_uid(sensing_dir / 'phonelock')
dark_f   = files_by_uid(sensing_dir / 'dark')
charge_f = files_by_uid(sensing_dir / 'phonecharge')
act_f    = files_by_uid(sensing_dir / 'activity')
app_f    = files_by_uid(DATA_ROOT / 'app_usage')

rows = []
problems = []
uids = sorted(answers['uid'].unique())
for k, uid in enumerate(uids, 1):
    try:
        ans = answers[answers['uid'] == uid]
        lock = read_intervals(lock_f[uid]); dark = read_intervals(dark_f[uid]); chg = read_intervals(charge_f[uid])
        a_ts, a_code = read_activity(act_f[uid])
        p_ts, p_code = read_apps(app_f[uid])

        # time grid: from 24h before the earliest timestamp to just after the latest
        all_min = min(lock['start'].min(), dark['start'].min(), chg['start'].min(), a_ts.min(), p_ts.min(), ans['resp_time'].min())
        all_max = max(lock['end'].max(), dark['end'].max(), chg['end'].max(), a_ts.max(), p_ts.max(), ans['resp_time'].max())
        t0 = int(all_min // 60 * 60) - WINDOW_MIN * 60
        n = int((all_max - t0) // 60) + 2

        m_lock, m_dark, m_chg = minute_mask(lock, t0, n), minute_mask(dark, t0, n), minute_mask(chg, t0, n)
        local_hour = pd.to_datetime(t0 + 60 * np.arange(n), unit='s', utc=True).tz_convert(TZ).hour
        night = np.asarray((local_hour >= 22) | (local_hour < 6))

        # cumulative counts per activity code -> fast window counts
        cum = {c: np.concatenate([[0], np.cumsum(a_code == c)]) for c in ACT_CODES}

        for r in ans.itertuples(index=False):
            hi = int((r.resp_time - t0) // 60)
            lo = hi - WINDOW_MIN
            t_lo, t_hi = r.resp_time - WINDOW_MIN * 60, r.resp_time
            row = {'uid': uid, 'resp_time': r.resp_time, 'stress_level_code': r.stress_level_code, 'y_stressed': r.y_stressed,
                   'lock_min': m_lock[lo:hi].sum(), 'dark_min': m_dark[lo:hi].sum(), 'charge_min': m_chg[lo:hi].sum(),
                   'night_lock_min': (m_lock & night)[lo:hi].sum(),
                   'night_dark_min': (m_dark & night)[lo:hi].sum(),
                   'night_charge_min': (m_chg & night)[lo:hi].sum()}
            i0, i1 = np.searchsorted(a_ts, t_lo, 'left'), np.searchsorted(a_ts, t_hi, 'left')
            tot = i1 - i0
            row['act_n'] = tot
            for c in ACT_CODES:
                row[f'act_frac_{c}'] = (cum[c][i1] - cum[c][i0]) / tot if tot > 0 else np.nan
            j0, j1 = np.searchsorted(p_ts, t_lo, 'left'), np.searchsorted(p_ts, t_hi, 'left')
            row['app_n_snapshots'] = np.unique(p_ts[j0:j1]).size
            row['app_n_distinct'] = np.unique(p_code[j0:j1]).size
            rows.append(row)
    except Exception as e:
        problems.append((uid, repr(e)))
    if k % 10 == 0:
        print(f'processed {k}/{len(uids)} participants')

table = pd.DataFrame(rows)
print('model table (before filtering):', table.shape, '| participants:', table['uid'].nunique())
print('participants that failed:', problems if problems else 'none')
display(table.head())

## Cell 5 - Keep only answers with real sensing, then add the personal-baseline features
- **Coverage rule:** an answer is kept only if there was at least 1 activity sample in its 24h window (`act_n > 0`). Otherwise "no data" would look like "no phone use". The number dropped is printed. (This threshold is a simple first rule, we can tighten it later.)
- **Baseline deviation:** `dev_<f>` = value minus the average of that person's EARLIER windows (needs 3 earlier answers). Only looks backward in time.

In [ ]:
before = len(table)
table = table[table['act_n'] > 0].copy()
print(f'dropped for no activity data in the 24h window: {before - len(table)} of {before}')

table = table.sort_values(['uid', 'resp_time']).reset_index(drop=True)
DEV_COLS = ['lock_min', 'dark_min', 'night_lock_min', 'act_frac_0', 'app_n_snapshots']
for c in DEV_COLS:
    prev_mean = table.groupby('uid')[c].transform(lambda s: s.shift(1).expanding(min_periods=3).mean())
    table[f'dev_{c}'] = table[c] - prev_mean

FEATURES = [c for c in table.columns if c not in ('uid', 'resp_time', 'stress_level_code', 'y_stressed')]
print('number of features:', len(FEATURES))
print(FEATURES)
print('\nanswers per participant (min / median / max):',
      table.groupby('uid').size().min(), table.groupby('uid').size().median(), table.groupby('uid').size().max())
print('\nclass balance after filtering:\n', table['y_stressed'].value_counts())
print('\nfraction missing per feature:')
display(table[FEATURES].isna().mean().round(3).sort_values(ascending=False).to_frame('missing'))

## Cell 6 - Participant-aware train/test split (no person in both)
Splits **people**, not rows: about 80% of participants go to train and 20% to test. Because one person's answers are very similar to each other, mixing them across train and test would give fake high scores.
The split is fixed (`random_state=42`) so everyone gets the same result, and the lists are saved.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(table, table['y_stressed'], groups=table['uid']))
train, test = table.iloc[train_idx], table.iloc[test_idx]

assert set(train['uid']).isdisjoint(set(test['uid'])), 'LEAK: a participant is in both sets!'
print('train: rows', len(train), '| participants', train['uid'].nunique())
print('test : rows', len(test),  '| participants', test['uid'].nunique())
print('\nclass balance (share of stressed):')
print('  train:', round(train['y_stressed'].mean(), 3), ' test:', round(test['y_stressed'].mean(), 3), ' all:', round(table['y_stressed'].mean(), 3))

## Cell 7 - Save
Saves the model table and the split lists to Drive for step 4.

In [ ]:
table.to_csv(OUT_DIR / 'studentlife_model_table.csv', index=False)
with open(OUT_DIR / 'studentlife_split.json', 'w') as fh:
    json.dump({'train_uids': sorted(train['uid'].unique().tolist()),
               'test_uids': sorted(test['uid'].unique().tolist()),
               'features': FEATURES, 'target': 'y_stressed',
               'notes': 'level 1 excluded; 24h window before answer; participant id never a feature'}, fh, indent=2)
print('saved: studentlife_model_table.csv and studentlife_split.json in', OUT_DIR)

## Cell 8 - What to send back to Claude
Send the printed output of Cells 2, 4, 5, 6. Next (step 4): train Logistic Regression (baseline) and Random Forest on the train people, and evaluate on the test people with accuracy, precision, recall, F1 and a confusion matrix.